In [8]:
from dotenv import load_dotenv
load_dotenv()

True

# State
Agent的短期记忆，存储当前会话的历史消息、任务状态等信息。

### 自定义State
定义一个类，继承AgentState，添加想要记录的属性信息

In [9]:
from langchain.agents import AgentState
from typing import NotRequired   # 用来标记字典中的某个键是可选的（即赋值时可以有也可以没有）。

# 自定义State结构
# 继承AgentState已经具备messages属性
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: NotRequired[int]  # 模型调用次数
    session_start: NotRequired[str]  # 会话开始时间

在工具中访问state  
LangChain内置了一个runtime参数，可以获取Agent的内部信息（state, store, context）  
runtime是LangChain中tool的限定参数，自定义参数不能叫这个名字。

In [10]:
from langchain.tools import tool, ToolRuntime

@tool
def my_tool(runtime: ToolRuntime):
    """a tool that update agent state"""
    pass

修改state：  
通过返回一个update格式的Command指令

In [11]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from datetime import datetime

@tool
def update_state(runtime: ToolRuntime):
    """a tool that update agent state"""
    # 获取state中的历史消息
    messages = runtime.state['messages']
    # 消息数量
    message_count = len(messages)
    # 组织结果
    command = {
        "model_call_count": runtime.state.get("model_call_count", 0) + 1,
        "messages": [ToolMessage("Successfully updated agent state", tool_call_id = runtime.tool_call_id)]
    }

    if message_count <= 2:
        command['session_start'] = datetime.now()

    return Command(update=command)

设置state schema

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "deepseek-chat",
    tools = [update_state],
    state_schema=CustomState,  # 使用自定义state
    checkpointer=InMemorySaver(),
    system_prompt="你是一名海盗，以海盗的口吻来和我说话。"
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="hello!")]},
    config
)

for message in response["messages"]:
    message.pretty_print()

In [ ]:
# 查看state消息
agent.get_state(config)